# Wikipedia Edits Data Pipeline - Solutions

This notebook demonstrates a simple ETL pipeline that:
1. Extracts data from the Wikipedia REST API
2. Transforms it to JSON Lines format
3. Uploads directly to S3

## Choose Your Username

Pick a unique username (e.g., your name or initials) and use it consistently:

- **S3 Bucket**: `<username>-wikidata` (e.g., `johndoe-wikidata`)
- **Athena Database**: `<username>` (e.g., `johndoe`)
- **Lambda**: Use the same `<username>-wikidata` bucket

**Important:** Use lowercase letters and digits only (e.g., `johndoe`): no hyphens, underscores or capitals. This one rule keeps the name valid for the S3 bucket, the Athena database and the Lambda function.

In [1]:
# Set your username here - use it consistently across all resources
USERNAME = "zoltanctothcicdtest"

## Setup and Imports

In [ ]:
import datetime
import json

import boto3
import requests

# The next line is only relevant for courseware development, please ignore it 
# and leave it commented out when working on this notebook
# boto3.setup_default_session(profile_name="ceu", region_name="eu-west-1")


## Extract: Retrieve Data from Wikipedia API

We use the Wikimedia Analytics API to fetch the most edited pages for a specific date. The API returns JSON with page titles and edit counts.

**API Documentation:** https://doc.wikimedia.org/generated-data-platform/aqs/analytics-api/reference/edits.html

In [3]:
# Try different dates to see how the data changes
DATE_PARAM = "2025-11-22"

date = datetime.datetime.strptime(DATE_PARAM, "%Y-%m-%d")

# Construct the API URL
url = f"https://wikimedia.org/api/rest_v1/metrics/edited-pages/top-by-edits/en.wikipedia/user/content/{date.strftime('%Y/%m/%d')}"
print(f"Requesting REST API URL: {url}")

# Make the API request
wiki_server_response = requests.get(url, headers={"User-Agent": "curl/7.68.0"})
wiki_response_status = wiki_server_response.status_code
wiki_response_body = wiki_server_response.text

print(f"Wikipedia REST API Response body: {wiki_response_body[:500]}...")
print(f"Wikipedia REST API Response Code: {wiki_response_status}")

# Validate response
if wiki_response_status != 200:
    raise Exception(f"Received non-OK status code from Wiki Server: {wiki_response_status}")
print(f"Successfully retrieved Wikipedia data, content-length: {len(wiki_response_body)}")

Requesting REST API URL: https://wikimedia.org/api/rest_v1/metrics/edited-pages/top-by-edits/en.wikipedia/user/content/2025/11/22
Wikipedia REST API Response body: {"items":[{"project":"en.wikipedia","editor-type":"user","page-type":"content","granularity":"daily","results":[{"timestamp":"2025-11-22T00:00:00.000Z","top":[{"page_title":"Silicon_shield","edits":215,"rank":1},{"page_title":"Duployan_shorthand","edits":134,"rank":2},{"page_title":"ABC_(band)","edits":131,"rank":3},{"page_title":"2026_AFC_U-17_Asian_Cup_qualification","edits":113,"rank":4},{"page_title":"Mike_Tilleman","edits":107,"rank":5},{"page_title":"The_First_Lady_(American_TV_series)","e...
Wikipedia REST API Response Code: 200
Successfully retrieved Wikipedia data, content-length: 6468


## Transform: Process Raw Data into JSON Lines

Convert the raw API response into a structured JSON Lines format suitable for analytics. Each line is a valid JSON object representing one page's edit statistics.

In [4]:
# Parse the API response and extract top edits
wiki_response_parsed = wiki_server_response.json()
top_edits = wiki_response_parsed["items"][0]["results"][0]["top"]

# Transform to JSON Lines format
current_time = datetime.datetime.now(datetime.timezone.utc)
json_lines = ""
for page in top_edits:
    record = {
        "title": page["page_title"],
        "edits": page["edits"],
        "date": date.strftime("%Y-%m-%d"),
        "retrieved_at": current_time.replace(tzinfo=None).isoformat(),
    }
    json_lines += json.dumps(record) + "\n"

print(f"Transformed {len(top_edits)} records to JSON Lines")
print(f"First few lines:\n{json_lines[:500]}...")

Transformed 100 records to JSON Lines
First few lines:
{"title": "Silicon_shield", "edits": 215, "date": "2025-11-22", "retrieved_at": "2025-12-09T08:48:44.461988"}
{"title": "Duployan_shorthand", "edits": 134, "date": "2025-11-22", "retrieved_at": "2025-12-09T08:48:44.461988"}
{"title": "ABC_(band)", "edits": 131, "date": "2025-11-22", "retrieved_at": "2025-12-09T08:48:44.461988"}
{"title": "2026_AFC_U-17_Asian_Cup_qualification", "edits": 113, "date": "2025-11-22", "retrieved_at": "2025-12-09T08:48:44.461988"}
{"title": "Mike_Tilleman", "edits": 1...


---
## Lab 1: Create an S3 Bucket

**Task:** Create an S3 bucket for the Wikipedia data pipeline.

**Requirements:**
- Bucket name: `<username>-wikidata` (use your USERNAME from above)
- Create the bucket if it doesn't exist

**Documentation:** [create_bucket](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/client/create_bucket.html)

In [5]:
S3_WIKI_BUCKET = f"{USERNAME}-wikidata"
s3 = boto3.client("s3", region_name="eu-west-1")

# Lab 1 Solution: Create the bucket if it doesn't exist
bucket_names = [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]]
if S3_WIKI_BUCKET not in bucket_names:
    s3.create_bucket(
        Bucket=S3_WIKI_BUCKET,
        CreateBucketConfiguration={"LocationConstraint": "eu-west-1"},
    )
    print(f"Created new bucket: {S3_WIKI_BUCKET}")
else:
    print(f"Using existing bucket: {S3_WIKI_BUCKET}")

Using existing bucket: zoltanctothcicdtest-wikidata


In [6]:
# Test Lab 1
assert USERNAME != "<username>", "Please set your USERNAME at the top of the notebook"
assert S3_WIKI_BUCKET.endswith("-wikidata"), "Bucket name must end with '-wikidata'"

try:
    s3.head_bucket(Bucket=S3_WIKI_BUCKET)
    print(f"Bucket {S3_WIKI_BUCKET} exists!")
except Exception as e:
    print(f"Bucket {S3_WIKI_BUCKET} not found: {e}")
    raise

Bucket zoltanctothcicdtest-wikidata exists!


---
## Lab 2: Upload JSON Lines to S3

**Task:** Upload the `json_lines` data directly to S3 (no local file!).

**Requirements:**
- Use `s3.put_object()` to upload the data directly
- Place the file under `raw-edits/` prefix in S3
- File name: `raw-edits-YYYY-MM-DD.json` (use the date from `DATE_PARAM`)

**Example S3 path:** `s3://johndoe-wikidata/raw-edits/raw-edits-2025-11-25.json`

**Documentation:** [put_object](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/s3/client/put_object.html)

In [7]:
# Lab 2 Solution: Upload json_lines directly to S3
s3_key = f"raw-edits/raw-edits-{date.strftime('%Y-%m-%d')}.json"
s3.put_object(
    Bucket=S3_WIKI_BUCKET,
    Key=s3_key,
    Body=json_lines,
)
print(f"Uploaded {len(top_edits)} records to s3://{S3_WIKI_BUCKET}/{s3_key}")

Uploaded 100 records to s3://zoltanctothcicdtest-wikidata/raw-edits/raw-edits-2025-11-22.json


In [8]:
# Test Lab 2
expected_key = f"raw-edits/raw-edits-{date.strftime('%Y-%m-%d')}.json"
try:
    s3.head_object(Bucket=S3_WIKI_BUCKET, Key=expected_key)
    print(f"File uploaded successfully to s3://{S3_WIKI_BUCKET}/{expected_key}")
except Exception as e:
    print(f"File not found at s3://{S3_WIKI_BUCKET}/{expected_key}")
    raise

File uploaded successfully to s3://zoltanctothcicdtest-wikidata/raw-edits/raw-edits-2025-11-22.json
